# Pipeline 7 - Donation Allocation Efficiency Predictor

Generated: 2026-04-08T16:56:17.331191Z


## 1) Problem Framing

**Business question:** Which donation allocations are most likely to produce stronger next-month safehouse outcomes per peso allocated?

**Who cares:** nonprofit leadership, operations coordinators, and donor reporting staff.

**Why it matters:** allocation decisions are one of the few levers leadership can control directly. Better allocation improves resident outcomes while strengthening donor trust through clearer impact stories.

**Predictive vs. explanatory goal:** this notebook includes both. The predictive model estimates allocation efficiency on unseen data. The explanatory model highlights which allocation patterns and contexts are associated with better outcomes. We do not interpret these relationships as automatic causal proof.

**Success metrics:** regression performance is evaluated with MAE, RMSE, and R-squared and compared against a baseline. Results are interpreted as decision-support signals for budget planning, not guaranteed effects.


## Notebook Setup

Shared imports and helper functions are defined once here so the later rubric sections can focus on pipeline-specific code.


In [ ]:
import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "ml-pipelines" else cwd
DATA_DIR = REPO_ROOT / "data" / "raw"
OUT_DIR = REPO_ROOT / "output" / "ml-predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data dir:", DATA_DIR)
print("Output dir:", OUT_DIR)

def require_csv(stem: str) -> pd.DataFrame:
    path = DATA_DIR / f"{stem}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the INTEX CSVs under data/raw/.")
    return pd.read_csv(path, encoding="utf-8-sig")

def numeric(series, fill_value=0.0):
    return pd.to_numeric(series, errors="coerce").fillna(fill_value)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def eval_regression(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse(y_true, y_pred),
        "r2": float(r2_score(y_true, y_pred)),
    }

def regression_baseline(y_train, y_test):
    median_value = float(pd.Series(y_train).median())
    pred = np.repeat(median_value, len(y_test))
    out = eval_regression(y_test, pred)
    out["baseline_value"] = median_value
    return out

def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def prep(cat_cols, num_cols):
    return ColumnTransformer([("cat", make_encoder(), cat_cols), ("num", "passthrough", num_cols)])

def top_features(pipe, n=10):
    model = pipe.named_steps["model"]
    names = pipe.named_steps["pre"].get_feature_names_out()
    if hasattr(model, "feature_importances_"):
        weights = np.asarray(model.feature_importances_)
    elif hasattr(model, "coef_"):
        weights = np.abs(np.asarray(model.coef_)).ravel()
    else:
        return pd.DataFrame()
    return pd.DataFrame({"feature": names, "importance": weights}).sort_values("importance", ascending=False).head(n).reset_index(drop=True)

def score_bands(scores):
    scores = pd.Series(scores).astype(float)
    if len(scores) == 0 or scores.nunique(dropna=True) < 2:
        return pd.Series(["Medium"] * len(scores), index=scores.index)
    labels = ["Low", "Medium", "High", "Very High"]
    q = min(4, scores.nunique(dropna=True), len(scores))
    try:
        return pd.qcut(scores.rank(method="first"), q=q, labels=labels[:q], duplicates="drop").astype(str)
    except Exception:
        return pd.Series(["Medium"] * len(scores), index=scores.index)

def export_predictions_json(prediction_type, entity_type, df_out, id_col, score_col, label_col=None):
    out_path = OUT_DIR / f"{prediction_type}.json"
    excluded = {id_col, score_col}
    if label_col:
        excluded.add(label_col)
    rows = []
    for _, row in df_out.iterrows():
        payload = {k: v for k, v in row.items() if k not in excluded}
        rows.append({
            "predictionType": prediction_type,
            "entityType": entity_type,
            "entityId": int(row[id_col]),
            "score": float(row[score_col]),
            "label": None if label_col is None or pd.isna(row[label_col]) else str(row[label_col]),
            "payloadJson": json.dumps(payload, default=str),
        })
    out_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print(f"Wrote {len(rows)} predictions:", out_path)


## 2) Data Acquisition, Preparation, and Exploration

This pipeline joins donations, donation allocations, safehouses, and safehouse monthly metrics to create an allocation-level dataset. The target estimates next-month safehouse improvement after allocation.


In [ ]:
donations = require_csv("donations")
alloc = require_csv("donation_allocations")
safehouses = require_csv("safehouses")
monthly = require_csv("safehouse_monthly_metrics")

donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
alloc["allocation_date"] = pd.to_datetime(alloc["allocation_date"], errors="coerce")
monthly["month_start"] = pd.to_datetime(monthly["month_start"], errors="coerce")

alloc["amount_allocated"] = numeric(alloc["amount_allocated"])
monthly["avg_education_progress"] = numeric(monthly["avg_education_progress"])
monthly["avg_health_score"] = numeric(monthly["avg_health_score"])
monthly["incident_count"] = numeric(monthly["incident_count"])
monthly["active_residents"] = numeric(monthly["active_residents"])

m = monthly[["safehouse_id","month_start","avg_education_progress","avg_health_score","incident_count","active_residents"]].copy()
m = m.sort_values(["safehouse_id","month_start"])
g = m.groupby("safehouse_id")
m["next_edu"] = g["avg_education_progress"].shift(-1)
m["next_health"] = g["avg_health_score"].shift(-1)
m["next_incidents"] = g["incident_count"].shift(-1)

m["edu_delta"] = m["next_edu"] - m["avg_education_progress"]
m["health_delta"] = m["next_health"] - m["avg_health_score"]
m["incident_delta"] = m["next_incidents"] - m["incident_count"]

# Composite outcome where larger is better.
m["next_month_outcome_gain"] = m["edu_delta"].fillna(0) + m["health_delta"].fillna(0) - 0.5 * m["incident_delta"].fillna(0)

alloc["month_start"] = alloc["allocation_date"].dt.to_period("M").dt.to_timestamp()

df = alloc.merge(
    donations[["donation_id","donation_type","channel_source","campaign_name","impact_unit","is_recurring"]],
    on="donation_id",
    how="left",
).merge(
    safehouses[["safehouse_id","region","status","capacity_girls","current_occupancy"]],
    on="safehouse_id",
    how="left",
).merge(
    m[["safehouse_id","month_start","avg_education_progress","avg_health_score","incident_count","active_residents","next_month_outcome_gain"]],
    on=["safehouse_id","month_start"],
    how="left",
)

df = df.dropna(subset=["next_month_outcome_gain"]).copy()
df["allocation_per_resident"] = df["amount_allocated"] / df["active_residents"].replace(0, np.nan)
df["occupancy_rate"] = df["current_occupancy"] / df["capacity_girls"].replace(0, np.nan)

df["allocation_per_resident"] = numeric(df["allocation_per_resident"]).fillna(0)
df["occupancy_rate"] = numeric(df["occupancy_rate"]).fillna(0)

cat_cols = ["program_area","donation_type","channel_source","campaign_name","impact_unit","region","status"]
num_cols = ["amount_allocated","allocation_per_resident","avg_education_progress","avg_health_score","incident_count","active_residents","occupancy_rate"]

for c in cat_cols:
    df[c] = df[c].fillna("Unknown")
for c in num_cols:
    df[c] = numeric(df[c])

target = "next_month_outcome_gain"
print("Rows:", len(df))
print(df[target].describe().to_string())


## 3) Modeling and Feature Selection

The predictive model uses a random forest regressor for nonlinear interactions. The explanatory model uses linear regression to provide easier coefficient interpretation.


In [ ]:
features = cat_cols + num_cols
X_train, X_test, y_train, y_test = train_test_split(df[features], df[target], test_size=0.25, random_state=42)

predictive = Pipeline([
    ("pre", prep(cat_cols, num_cols)),
    ("model", RandomForestRegressor(n_estimators=250, random_state=42, min_samples_leaf=3)),
])

explanatory = Pipeline([
    ("pre", prep(cat_cols, num_cols)),
    ("model", LinearRegression()),
])


## 4) Evaluation and Selection

The model is evaluated on a holdout split and compared against a baseline to avoid overclaiming value.


In [ ]:
print("Baseline:", regression_baseline(y_train, y_test))

predictive.fit(X_train, y_train)
pred = predictive.predict(X_test)
print("Predictive:", eval_regression(y_test, pred))

explanatory.fit(X_train, y_train)
pred_exp = explanatory.predict(X_test)
print("Explanatory:", eval_regression(y_test, pred_exp))


## 5) Causal and Relationship Analysis

This section highlights major observed relationships. These are correlational findings and should be validated through policy trials or controlled process changes before making high-stakes decisions.


In [ ]:
print("Top predictive features:")
print(top_features(predictive).to_string(index=False))

print("Top explanatory features (absolute coefficients):")
print(top_features(explanatory).to_string(index=False))

print("Business takeaway: prioritize allocation patterns with consistently positive predicted gain, but require human review and monitor fairness across safehouses.")


## 6) Deployment Notes

The exported JSON can be imported via `POST /api/ml/import?replace=true`. In the app, this can be shown as an allocation planning signal in ML Insights and operations reports.


In [ ]:
df_out = df[["allocation_id","safehouse_id","program_area","amount_allocated","allocation_per_resident","occupancy_rate","avg_education_progress","avg_health_score","incident_count"]].copy()
df_out["predicted_allocation_gain"] = predictive.predict(df[features])
df_out["allocation_band"] = score_bands(df_out["predicted_allocation_gain"])

export_predictions_json(
    "allocation_efficiency_gain",
    "Allocation",
    df_out,
    "allocation_id",
    "predicted_allocation_gain",
    "allocation_band",
)
